In [5]:
import pandas as pd

from config.global_config import ModelType
from predictions.predict_dataset import predict_dataset

dummy_data = pd.read_csv('statics/datasets/training.csv')

predicted_df = predict_dataset(dummy_data, ModelType.FINE_TUNED)
      

In [ ]:
import pandas as pd

from config.global_config import ModelType
from predictions.predict_dataset import predict_dataset

predict_dataset(dummy_data, ModelType.ZERO_SHOT)

In [ ]:
import pandas as pd

from config.global_config import ModelType
from predictions.predict_dataset import predict_dataset

predict_dataset(dummy_data, ModelType.LLM)

In [7]:
df = pd.read_csv("statics/datasets/training.csv")
df.head(), len(df)

(   annotation_id  annotator   attractions        category   cleanliness  \
 0              1          2  notmentioned            Park      negative   
 1              2          2  notmentioned            Park      negative   
 2              3          2  notmentioned  Parking garage  notmentioned   
 3              4          2  notmentioned  Parking garage  notmentioned   
 4              5          2  notmentioned  Parking garage  notmentioned   
 
           costs                   created_at      heritage  id infrastructure  \
 0  notmentioned  2026-04-03T18:08:05.212014Z  notmentioned   1   notmentioned   
 1  notmentioned  2026-04-03T18:08:49.682261Z  notmentioned   2   notmentioned   
 2  notmentioned  2026-04-03T18:09:15.153045Z  notmentioned   3   notmentioned   
 3      positive  2026-04-03T18:10:13.194366Z  notmentioned   4   notmentioned   
 4      positive  2026-04-03T18:10:39.551301Z  notmentioned   5        neutral   
 
    ...  lead_time  longitude  \
 0  ...     29.

In [8]:
safety_accuracy = (df['safety'] == predicted_df['safety']).mean()

print(f"Accuracy for Safety: {safety_accuracy:.2%}")

cleanliness_accuracy = (df['cleanliness'] == predicted_df['cleanliness']).mean()

print(f"Accuracy for Cleanliness: {cleanliness_accuracy:.2%}")

strict_match = (df['safety'] == predicted_df['safety']) & \
               (df['cleanliness'] == predicted_df['cleanliness'])

strict_accuracy = strict_match.mean()
print(f"Strict Accuracy (Both must match): {strict_accuracy:.2%}")

average_accuracy = (safety_accuracy + cleanliness_accuracy) / 2
print(f"Average Accuracy across both aspects: {average_accuracy:.2%}")

Accuracy for Safety: 92.10%
Accuracy for Cleanliness: 88.40%
Strict Accuracy (Both must match): 81.40%
Average Accuracy across both aspects: 90.25%


In [13]:
from application.dataset_types import DatasetType
from application.results_repository import EntryMetadata, ResultsRepository
from config.global_config import ModelType

results_repository = ResultsRepository()
csv_bytes = predicted_df.to_csv(index=False).encode('utf-8')

results_repository.save(
    "labelled_reviews.csv", 
    csv_bytes, 
    EntryMetadata(
        csv_filename="labelled_reviews.csv",
        dataset_type=DatasetType.LABELLED_AI, 
        model_type=ModelType.FINE_TUNED
        )
    )

PosixPath('statics/results_repository/labelled_reviews.csv')

In [1]:
import importlib
import predictions.prediction_fine_tuned
importlib.reload(predictions.prediction_fine_tuned)

from config.global_config import TRAIN_ASPECTS
from predictions.prediction_fine_tuned import FineTunedModel

# Load model
model = FineTunedModel(
    local_model_path='saved_models/bert-base-uncased_absa.pt',
    aspects=TRAIN_ASPECTS,
)
print('Model loaded successfully!')
print(f'Aspects: {model.aspects}')

# Test predictions
test_texts = [
    'This park is dirty and unsafe at night, the paths are broken.',
    'Beautiful museum with free entry and amazing historical exhibitions!',
    'The place was clean but nothing special.',
]

for text in test_texts:
    result = model.predict(text)
    active = {k: v for k, v in result.items() if v != 'notmentioned'}
    print(f'\n>>> {text}')
    print(f'  Active: {active}')
    print(f'  Full: {result}')

/Users/bartoszbugla/projects/projekt-magisterski/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17778.91it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!
Aspects: ['safety', 'cleanliness', 'infrastructure', 'nature', 'attractions', 'heritage', 'costs', 'other']

>>> This park is dirty and unsafe at night, the paths are broken.
  Active: {'safety': 'negative', 'cleanliness': 'negative', 'infrastructure': 'negative'}
  Full: {'safety': 'negative', 'cleanliness': 'negative', 'infrastructure': 'negative', 'nature': 'notmentioned', 'attractions': 'notmentioned', 'heritage': 'notmentioned', 'costs': 'notmentioned', 'other': 'notmentioned'}

>>> Beautiful museum with free entry and amazing historical exhibitions!
  Active: {'safety': 'positive', 'infrastructure': 'positive', 'attractions': 'positive', 'other': 'positive'}
  Full: {'safety': 'positive', 'cleanliness': 'notmentioned', 'infrastructure': 'positive', 'nature': 'notmentioned', 'attractions': 'positive', 'heritage': 'notmentioned', 'costs': 'notmentioned', 'other': 'positive'}

>>> The place was clean but nothing special.
  Active: {'cleanliness': 'neutral'